In [18]:
import matplotlib.pyplot as plt 
import os 
import re 
import shutil 
import string 
import tensorflow as tf 
from tensorflow.keras import layers

In [19]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

In [20]:
dataset = tf.keras.utils.get_file( "aclImdb_v1", url, untar=True, cache_dir='.', cache_subdir='' ) 
dataset_dir = os.path.join(os.getcwd(),"aclImdb_v1", "aclImdb")

In [21]:
os.listdir(dataset_dir) 

['.ipynb_checkpoints', 'imdb.vocab', 'imdbEr.txt', 'README', 'test', 'train']

In [22]:
train_dir = os.path.join(dataset_dir,"train") 
os.listdir(train_dir) 

['labeledBow.feat',
 'neg',
 'pos',
 'unsup',
 'unsupBow.feat',
 'urls_neg.txt',
 'urls_pos.txt',
 'urls_unsup.txt']

In [23]:
shutil.rmtree(os.path.join(train_dir,"unsup"))

In [24]:
batch_size = 32 
seed = 42 
raw_train_ds = tf.keras.utils.text_dataset_from_directory( 
    train_dir, 
    batch_size=batch_size, 
    validation_split=0.2, 
    subset='training', 
    seed=seed)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.


In [25]:
raw_val_ds = tf.keras.utils.text_dataset_from_directory( 
    train_dir, 
    batch_size=batch_size, 
    validation_split=0.2, 
    subset='validation', 
    seed=seed) 

Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [26]:
test_dir = os.path.join(dataset_dir,"test") 
os.listdir(test_dir) 

['labeledBow.feat', 'neg', 'pos', 'urls_neg.txt', 'urls_pos.txt']

In [27]:
raw_test_ds = tf.keras.utils.text_dataset_from_directory( 
    test_dir, 
    batch_size=batch_size) 

Found 25000 files belonging to 2 classes.


In [28]:
def custom_standardization(input_data): 
    lowercase = tf.strings.lower(input_data) 
    stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ') 
    return tf.strings.regex_replace( 
        stripped_html, 
        f'[{re.escape(string.punctuation)}]', 
        '' 
    )

In [29]:
max_features = 10000 
sequence_length = 250 
vectorize_layer = layers.TextVectorization( 
    standardize=custom_standardization, 
    max_tokens=max_features, 
    output_mode='int', 
    output_sequence_length=sequence_length) 
train_text = raw_train_ds.map(lambda x, y: x) 
vectorize_layer.adapt(train_text) 
def vectorize_text(text,label): 
    text = tf.expand_dims(text,-1) 
    return vectorize_layer(text),label 

In [30]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE) 
val_ds = raw_val_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE) 
test_ds = raw_test_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE)

In [31]:
embedding_dim = 16 
model = tf.keras.Sequential([ 
    layers.Embedding(max_features + 1, embedding_dim), 
    layers.Conv1D(8,7,activation="relu"), 
    layers.GlobalAveragePooling1D(), 
    layers.Dropout(0.2), 
    layers.Dense(8,activation="relu"), 
    layers.Dense(1)]) 
model.summary() 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.compile( 
    optimizer='adam', 
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True), 
    metrics=['accuracy'] 
) 
model.fit( 
    train_ds, 
    validation_data=val_ds, 
    epochs=10, 
    callbacks=[ 
        tf.keras.callbacks.TensorBoard(log_dir="logs") 
    ] 
) 

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.6866 - loss: 0.5196 - val_accuracy: 0.8442 - val_loss: 0.3612
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8494 - loss: 0.3403 - val_accuracy: 0.8656 - val_loss: 0.3194
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8755 - loss: 0.2904 - val_accuracy: 0.8656 - val_loss: 0.3074
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8960 - loss: 0.2490 - val_accuracy: 0.8718 - val_loss: 0.3031
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9087 - loss: 0.2235 - val_accuracy: 0.8670 - val_loss: 0.3101
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9178 - loss: 0.2059 - val_accuracy: 0.8674 - val_loss: 0.3233
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9255 - loss: 0.1850 - val_accuracy: 0.8646 - val_loss: 0.3360
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9317 - loss: 0.1714 - val_accuracy: 

In [33]:
%load_ext tensorboard 
%tensorboard --logdir logs

ERROR: Timed out waiting for TensorBoard to start. It may still be running as pid 6360.

In [34]:
loss, accuracy = model.evaluate(test_ds) 
print("Loss: ", loss) 
print("Accuracy: ", accuracy) 

782/782 ━━━━━━━━━━━━━━━━━━━━ 218s 274ms/step - accuracy: 0.8393 - loss: 0.4316
Loss:  0.4316035807132721
Accuracy:  0.8392800092697144


In [35]:
export_model = tf.keras.Sequential([ 
    vectorize_layer, 
    model, 
    layers.Activation('sigmoid') 
]) 

export_model.compile( 
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), 
    optimizer="adam", metrics=['accuracy'] 
) 
 
loss, accuracy = export_model.evaluate(raw_test_ds) 
print(accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 46s 57ms/step - accuracy: 0.8380 - loss: 0.4316  
0.8380399942398071


In [40]:
export_model.save("sentence_classification_model.h5") 